# Notebook 2 — SQL and Python Functions

This notebook does not use AI. First learn to retrieve exact facts from tables.
Run Notebook 1 first, and use the same catalog and schema.


In [0]:
catalog_name = "main"  # Keep consistent with Notebook 1
schema_name = "scm_e2e_rag_lab"
item_table = f"{catalog_name}.{schema_name}.item_master"
inventory_table = f"{catalog_name}.{schema_name}.inventory"
supplier_table = f"{catalog_name}.{schema_name}.supplier"


## 1. SQL example: How many SSDs are in Chennai?

Expected answer: 50.

In [0]:


display(spark.sql(f"""
SELECT i.item_name, v.warehouse, v.available_quantity
FROM {item_table} i
JOIN {inventory_table} v ON i.item_id = v.item_id
WHERE i.item_id = 'SSD001' AND v.warehouse = 'Chennai'
"""))


item_name,warehouse,available_quantity
1 TB NVMe SSD,Chennai,50


## 2. Function: get inventory for an item and warehouse

A function is reusable Python code. Here, it runs a database query and returns the matching rows.


In [0]:

def get_inventory(item_id: str, warehouse: str):
    # Use SQL parameter markers for values; table names are fixed above.
    query = f"""
        SELECT item_id, warehouse, available_quantity
        FROM {inventory_table}
        WHERE item_id = :item_id AND warehouse = :warehouse
    """
    return spark.sql(query, args={"item_id": item_id, "warehouse": warehouse}).collect()
print(get_inventory("SSD001", "Chennai"))
print(get_inventory("BAT001", "Chennai"))  # No matching row in this sample


[Row(item_id='SSD001', warehouse='Chennai', available_quantity=50)]
[]


## 3. Function: get supplier details


In [0]:

def get_supplier(item_id: str):
    query = f"""
        SELECT item_id, supplier_name, supplier_city, supplier_country, lead_time_days
        FROM {supplier_table}
        WHERE item_id = :item_id
    """
    return spark.sql(query, args={"item_id": item_id}).collect()
print(get_supplier("SSD001"))  # Includes supplier location: Chennai, India


[Row(item_id='SSD001', supplier_name='ABC Components', supplier_city='Chennai', supplier_country='India', lead_time_days=7)]


## 4. Practice questions

1. How many RAM002 items are available in Chennai?
2. Who supplies RAM002, where are they located, and what is the lead time?
3. Where is BAT001 supplier based? Is that the same as the warehouse location?
4. What happens when you search for BAT001 in the Chennai warehouse?

**Key lesson:** SQL/functions are best for exact values like quantity, supplier, and lead time.
